# Two-Stage Position Model v2

**v1 대비 개선사항**:
1. Feature Selection을 unit-level 집계 후로 이동 (die-level pseudo-replication 해결)
2. Val을 early stopping에 사용하지 않음 (K-Fold CV로 대체)
3. True Two-Stage 구현 (Y>0만 회귀) + 기존 방식 비교
4. CV 집계 수치 안정화 (clip 임계값 1e-10 → 1.0)
5. 집계 함수에 mean, std 추가 (총 7종)
6. Position별 피벗 feature 추가

## 1. 환경 설정 및 데이터 로드

In [1]:
# ============================================================
# 환경 설정 + 데이터 로드
# ============================================================
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    if not os.path.exists("/content/project/2_preprocessing/cleaning.py"):
        os.system("gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip")
        os.system("unzip -qo /content/preprocessing.zip -d /content/project")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# --- 기본 라이브러리 ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm

# --- 모델 ---
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold

# --- 프로젝트 설정 유틸 ---
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, POSITION_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import evaluate, postprocess, compare_models
from utils.experiment import log_experiment, check_exp_id, download_from_drive

# --- 전처리 모듈 (2_preprocessing/) ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, "2_preprocessing"))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from feature_selection import run_feature_selection, DEVICE

# --- 원본 데이터 로드 ---
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
ys_train = ys["train"]

print(f"LightGBM device: {DEVICE}")

setup 완료
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749
LightGBM device: gpu


In [ ]:
# ============================================================
# 실험 설정 값
# ============================================================

EXP_ID = "3-1-003"
EXP_TYPE = "Two-Stage v2 (unit-level FS + K-Fold + position pivot)"
EXP_MEMO = "v1 대비: unit-level FS, CV clip 안정화, mean/std/position pivot 추가, K-Fold CV, True Two-Stage vs Single 비교"

# --- Google Drive 파일 ID ---
XLSX_GDRIVE_ID = "1IgaNh7ixgqpmH5PiwmSFbK2li6GODdew"
JSON_GDRIVE_ID = "1ycr6n5Ty_jzl4F-qQE4Cv5nS2WbIAZih"

# --- 전처리: 클리닝 ---
cleaning_params = dict(
    const_threshold=1e-6,         # 분산이 이 값 이하인 상수 feature 제거
    missing_threshold=0.25,       # 결측률이 이 비율 이상인 feature 제거
    remove_duplicates=True,       # 값이 완전히 동일한 중복 컬럼 제거
    corr_threshold=0.95,          # 피처 간 |상관계수| > 이 값이면 한쪽 제거 (다중공선성 감소)
    add_indicator=True,           # 결측 여부를 나타내는 이진 indicator 컬럼 추가
    indicator_threshold=0.01,     # indicator 추가 기준 최소 결측률 (이 이상일 때만 생성)
    imputation_method="median",   # 결측 대체 방법: "median" (중앙값) | "knn" (k-최근접이웃)
    # knn_neighbors=5,              # knn imputation 시 참조할 이웃 수
)

# --- 전처리: 이상치 ---
outlier_params = dict(
    method="winsorize",            # 이상치 처리 방법: "winsorize" (분위수 clip) | "iqr_clip" (IQR 기반) | "none"
    lower_pct=0.01,               # winsorize 하위 분위수 (이 백분위 이하 값을 경계값으로 대체)
    upper_pct=0.995,               # winsorize 상위 분위수 (이 백분위 이상 값을 경계값으로 대체)
    iqr_multiplier=1.5,           # iqr_clip 시 IQR 배수 (Q1 - 1.5*IQR ~ Q3 + 1.5*IQR)
)

# --- Feature Selection ---
selection_params = dict(
    methods=["boruta", "lgbm_importance", "null_importance", "permutation"],  # 실행할 FS 방법 목록
    min_votes=3,          # 최종 채택 기준: 최소 N개 방법에서 동시 선택된 feature만 통과
    sample_n=None,        # Boruta용 서브샘플 수. None=전체 사용 (unit-level 26,247개)
    boruta_params=dict(
        max_iter=100,     # Boruta 최대 반복 횟수 (높을수록 정밀, 100~200 권장)
        max_depth=5,      # 내부 RandomForest의 트리 최대 깊이
        perc=80,          # Shadow feature 임계 백분위수 (80~100, 높을수록 엄격하게 선별)
    ),
    lgbm_params=dict(
        threshold=5,      # importance > threshold인 feature 선택 (0=기여하는 모든 feature)
    ),
    null_params=dict(
        n_runs=10,        # target 셔플 반복 횟수 (10~20 권장, 높을수록 안정적)
        threshold=2.0,    # z-score 기준 (실제 imp - 셔플 imp 평균) / 셔플 imp 표준편차 > 이 값
    ),
    rfe_params=dict(                   # methods에 "rfe" 추가 시 활성화
        n_features_to_select=100,  # RFE 최종 선택할 feature 수
        step=50,                   # 매 반복마다 제거할 feature 수
    ),
    perm_params=dict(                 # methods에 "permutation" 추가 시 활성화
        threshold=5,      # permutation importance > threshold인 feature 선택
        n_repeats=5,      # feature 셔플 반복 횟수
    ),
    mi_params=dict(                     # methods에 "mutual_info" 추가 시 활성화
        threshold=0,      # mutual information > threshold인 feature 선택
    ),
)

# --- 샘플링 ---
USE_SAMPLING = True               # True: 샘플링 사용, False: 전체 데이터
SAMPLE_FRAC = 0.2                 # 샘플링 비율 (unit 단위)

# --- Die→Unit 집계 ---
AGG_FUNCS = ["mean", "std", "cv", "range", "min", "max", "median"]  # v2: mean, std 추가
PROBA_AGG = "mean"                # 분류 확률 집계: "mean" | "max" | "median"

# --- Stage 1: 분류기 (이진분류 OOF) ---
clf_params = dict(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    # is_unbalanced=True,          # 클래스 불균형 자동 보정
    # scale_pos_weight=2.42,       # Y>0 가중치 (0.708/0.292)
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
    device=DEVICE,
)

# --- Stage 2: 회귀기 (unit-level 회귀) ---
reg_params = dict(
    objective="regression",          # "regression" | "poisson" | "tweedie"
    # tweedie_variance_power=1.5,   # tweedie 시 1.0~2.0
    n_estimators=10000,
    learning_rate=0.012457466004205876,
    num_leaves=49,
    max_depth=7,
    min_child_samples=245,
    subsample=0.7591420347752385,
    colsample_bytree=0.10415361830684666,
    reg_alpha=0.02321388377387803,
    reg_lambda=0.14505131037326144,
    min_split_gain=0.00010132317378668888,
    path_smooth=20.64149821542759,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
    device=DEVICE,
)

# --- OOF / Early Stopping ---
N_FOLDS = 5
CLF_EARLY_STOP = 50   # 분류기 patience
REG_EARLY_STOP = 50   # 회귀기 patience
LABEL_COL = "label_bin"

# --- 후처리 ---
CLF_THRESHOLD = 0.5   # P(Y>0) 이진 판별 임계값 (튜닝용)

print(f"실험번호: {EXP_ID} | {EXP_TYPE}")
print(f"집계 함수: {AGG_FUNCS} | 확률 집계: {PROBA_AGG}")
print(f"샘플링: {'ON' if USE_SAMPLING else 'OFF'} (frac={SAMPLE_FRAC})")
print(f"이상치: {outlier_params['method']} | Imputation: {cleaning_params['imputation_method']}")
print(f"회귀 objective: {reg_params['objective']}")
print(f"Device: {DEVICE}")

download_from_drive(XLSX_GDRIVE_ID, JSON_GDRIVE_ID)
check_exp_id(EXP_ID)

실험번호: 3-1-003 | Two-Stage v2 (unit-level FS + K-Fold + position pivot)
집계 함수: ['mean', 'std', 'cv', 'range', 'min', 'max', 'median'] | 확률 집계: mean
샘플링: ON (frac=0.2)
이상치: winsorize | Imputation: median
회귀 objective: regression
Device: gpu
  Drive → xlsx 다운로드 중...
  완료: /content/project/4_output/experiments/experiments.xlsx
  Drive → json 다운로드 중...
  완료: /content/project/4_output/experiments/experiments.json


## 2. 전처리 (클리닝 + 이상치)

In [3]:
# --- Step 1: 클리닝 ---
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    **cleaning_params,
)

클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=25%
  제거: 10개, 잔여: 972개
    컬럼: 982 → 972 (10개 제거)
    DataFrame: (104988, 976)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 946개
    컬럼: 972 → 946 (26개 제거)
    DataFrame: (104988, 950)

[고상관 제거] threshold=0.95
  제거: 212개, 잔여: 734개
    컬럼: 946 → 734 (212개 제거)
    DataFrame: (104988, 738)

[결측 indicator] 7개 컬럼 추가 (결측률 >= 1%)
[결측 imputation] method=median
  imputation 후 잔여 결측: 0

클리닝 완료: 1087 → 734 features (353개 제거)
  + indicator 컬럼: 7개 → 총 741개
  train: (104988, 745)
  val:   (34996, 745)
  test:  (34996, 745)


In [4]:
# --- Step 2: 이상치 처리 (Winsorization) ---
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    **outlier_params,
)

이상치 처리 파이프라인 시작 (method=winsorize)
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 129개
  이상치 > 10%: 64개
[Winsorization] lower=1%, upper=100%
  적용 feature: 741개
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 129개
  이상치 > 10%: 64개

이상치 처리 완료
  이상치 >5%  feature: 129 → 129 (0개 감소)
  이상치 >10% feature: 64 → 64 (0개 감소)
  train: (104988, 745)


## 3. Die-level Health Merge + 이진 라벨 + Position 분리

- Health(unit-level)를 die-level로 merge
- 이진 라벨: 0 vs >0 (분류기용)
- Position 1~4로 분리

In [5]:
# ============================================================
# Die-level에 Health(target) merge
# ============================================================
die_train = xs_train.merge(ys["train"], on=KEY_COL, how="left")
die_val = xs_val.merge(ys["validation"], on=KEY_COL, how="left")
die_test = xs_test.merge(ys["test"], on=KEY_COL, how="left")

assert die_train[TARGET_COL].notna().all(), "train health에 NaN!"
assert die_val[TARGET_COL].notna().all(), "val health에 NaN!"
assert die_test[TARGET_COL].notna().all(), "test health에 NaN!"
print(f"Die-level merge: train={die_train.shape}, val={die_val.shape}, test={die_test.shape}")

# ============================================================
# 샘플링 (unit 단위, train만)
# ============================================================
if USE_SAMPLING:
    all_units = die_train[KEY_COL].drop_duplicates()
    sampled_units = all_units.sample(frac=SAMPLE_FRAC, random_state=SEED)
    n_before = len(die_train)
    die_train = die_train[die_train[KEY_COL].isin(sampled_units)].reset_index(drop=True)
    print(f"
샘플링 ON (frac={SAMPLE_FRAC})")
    print(f"  Unit: {len(all_units):,} → {len(sampled_units):,}")
    print(f"  Die: {n_before:,} → {len(die_train):,}")
else:
    print("
샘플링 OFF → 전체 데이터 사용")

# -- 이진 라벨 생성: 0 vs >0 --
for df in [die_train, die_val, die_test]:
    df[LABEL_COL] = (df[TARGET_COL] > 0).astype(int)

print("
이진 라벨 분포 (train):")
dist = die_train[LABEL_COL].value_counts().sort_index()
for k, v in dist.items():
    pct = v / len(die_train) * 100
    print(f"  {k}: {v:,} ({pct:.1f}%)")

# ============================================================
# Position별 분리 (1 unit = 4 dies, position 1~4)
# ============================================================
positions = sorted(die_train[POSITION_COL].unique())
feat_cols_clean = clean_cols

pos_data = {}
for pos in positions:
    pos_data[pos] = {
        "train": die_train[die_train[POSITION_COL] == pos].reset_index(drop=True),
        "val":   die_val[die_val[POSITION_COL] == pos].reset_index(drop=True),
        "test":  die_test[die_test[POSITION_COL] == pos].reset_index(drop=True),
    }
    n = {k: len(v) for k, v in pos_data[pos].items()}
    print(f"Position {pos}: train={n['train']:,}, val={n['val']:,}, test={n['test']:,}")

print(f"
Clean feature 수: {len(feat_cols_clean)}개")

Die-level merge: train=(104988, 746), val=(34996, 746), test=(34996, 746)

이진 라벨 분포 (train):
  0: 74,332 (70.8%)
  1: 30,656 (29.2%)
Position 1: train=26,247, val=8,749, test=8,749
Position 2: train=26,247, val=8,749, test=8,749
Position 3: train=26,247, val=8,749, test=8,749
Position 4: train=26,247, val=8,749, test=8,749

Clean feature 수: 741개


## 4. Stage 1: Position별 이진분류 (OOF)

- Position마다 LightGBM 이진분류기를 K-Fold OOF로 학습
- 각 die에 대한 P(Y>0) 확률을 산출

In [7]:
def run_classification(pos_data, feat_cols, clf_params, n_folds=N_FOLDS):
    """Position별 이진분류 OOF 후 P(Y>0) 확률 반환"""
    all_results = {}

    for pos in sorted(pos_data.keys()):
        d = pos_data[pos]
        X_tr = d["train"][feat_cols].values
        y_tr = d["train"][LABEL_COL].values
        X_val = d["val"][feat_cols].values
        X_test = d["test"][feat_cols].values

        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
        oof_proba = np.zeros(len(X_tr))
        fold_models = []

        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            clf = lgb.LGBMClassifier(**clf_params)
            clf.fit(
                X_tr[tr_idx], y_tr[tr_idx],
                eval_set=[(X_tr[val_idx], y_tr[val_idx])],
                callbacks=[lgb.early_stopping(CLF_EARLY_STOP, verbose=False),
                           lgb.log_evaluation(0)],
            )
            oof_proba[val_idx] = clf.predict_proba(X_tr[val_idx])[:, 1]
            fold_models.append(clf)

        train_proba = oof_proba
        val_proba = np.mean([m.predict_proba(X_val)[:, 1] for m in fold_models], axis=0)
        test_proba = np.mean([m.predict_proba(X_test)[:, 1] for m in fold_models], axis=0)

        train_acc = ((train_proba > CLF_THRESHOLD).astype(int) == y_tr).mean()
        val_acc = ((val_proba > CLF_THRESHOLD).astype(int) == d["val"][LABEL_COL].values).mean()
        print(f"  Position {pos}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

        all_results[pos] = {
            "train_proba": train_proba,
            "val_proba": val_proba,
            "test_proba": test_proba,
        }

    return all_results


print("Stage 1: Position별 이진분류 (OOF)")
print("=" * 50)
clf_result = run_classification(pos_data, feat_cols_clean, clf_params)

Stage 1: Position별 이진분류 (OOF)
  Position 1: train_acc=0.7078, val_acc=0.7081
  Position 2: train_acc=0.7067, val_acc=0.7076
  Position 3: train_acc=0.7078, val_acc=0.7082
  Position 4: train_acc=0.7058, val_acc=0.7076


## 5. Die → Unit 집계

**v1 대비 변경**:
- 집계 함수: mean, std 추가 (총 7종)
- CV clip 임계값: 1e-10 → 1.0 (수치 안정화)
- Position별 피벗 feature 추가 (각 die의 원본 값 보존)

In [8]:
def aggregate_die_to_unit(pos_data, feat_cols, clf_result, agg_funcs=AGG_FUNCS):
    """Die-level -> Unit-level 집계. WT feature + 분류 확률을 unit 단위로 합산."""
    results = {}

    for split_name in ["train", "val", "test"]:
        die_frames = []
        for pos in sorted(pos_data.keys()):
            df = pos_data[pos][split_name].copy()
            df["clf_proba"] = clf_result[pos][f"{split_name}_proba"]
            die_frames.append(df)

        die_all = pd.concat(die_frames, ignore_index=True)

        # --- WT feature 집계 ---
        agg_dict = {}
        grp = die_all.groupby(KEY_COL)[feat_cols]

        for func in agg_funcs:
            if func == "cv":
                cv_df = grp.std() / grp.mean().abs().clip(lower=1.0)
                cv_df.columns = [f"{c}_cv" for c in feat_cols]
                agg_dict["cv"] = cv_df
            elif func == "range":
                range_df = grp.max() - grp.min()
                range_df.columns = [f"{c}_range" for c in feat_cols]
                agg_dict["range"] = range_df
            else:
                grp_df = grp.agg(func)
                grp_df.columns = [f"{c}_{func}" for c in feat_cols]
                agg_dict[func] = grp_df

        unit_features = pd.concat(agg_dict.values(), axis=1)

        # --- Position별 피벗 (각 die의 원본 값 보존) ---
        for pos in sorted(pos_data.keys()):
            pos_mask = die_all[POSITION_COL] == pos
            pos_df = die_all.loc[pos_mask].set_index(KEY_COL)[feat_cols]
            pos_df.columns = [f"{c}_pos{pos}" for c in feat_cols]
            unit_features = unit_features.join(pos_df)

        # --- 분류 확률 집계 (평균) ---
        proba_mean = die_all.groupby(KEY_COL)["clf_proba"].mean()
        proba_mean.name = "clf_proba_mean"
        unit_features = unit_features.join(proba_mean)

        # --- health (target) ---
        health = die_all.groupby(KEY_COL)[TARGET_COL].first()
        unit_features = unit_features.join(health)

        unit_features = unit_features.reset_index()
        results[split_name] = unit_features

    return results


print("Die -> Unit 집계")
print("=" * 50)
unit_data = aggregate_die_to_unit(pos_data, feat_cols_clean, clf_result)

unit_feat_cols = [c for c in unit_data["train"].columns
                  if c not in [KEY_COL, TARGET_COL]]

for sp in ["train", "val", "test"]:
    print(f"  {sp}: {unit_data[sp].shape}")
print(f"  Unit-level feature 수: {len(unit_feat_cols)}개")

Die -> Unit 집계
  train: (26247, 8154)
  val: (8749, 8154)
  test: (8749, 8154)
  Unit-level feature 수: 8152개


## 5.5 Feature Selection (Unit-level)

**v1 대비 변경**: die-level → unit-level로 이동
- 모델이 실제 사용하는 unit-level 집계 데이터에서 feature 중요도 평가
- die-level의 pseudo-replication (동일 target 4회 반복) 문제 해결

> feature 수가 ~10,000개로 Boruta에 시간이 걸릴 수 있음

In [ ]:
# --- 사전 필터: 분산 0인 unit feature 제거 ---
var = unit_data["train"][unit_feat_cols].var()
nonzero_cols = var[var > 0].index.tolist()
zero_var_removed = len(unit_feat_cols) - len(nonzero_cols)
print(f"Zero-variance unit feature 제거: {zero_var_removed}개")
print(f"Feature Selection 입력: {len(nonzero_cols)}개")

# --- Feature Selection (Boruta + LightGBM + Null Importance 투표) ---
selected_cols, sel_report = run_feature_selection(
    X_train=unit_data["train"],
    y_train=unit_data["train"][TARGET_COL],
    feat_cols=nonzero_cols,
    **selection_params,
)

n_before = len(unit_feat_cols)
unit_feat_cols = selected_cols
print(f"\nFeature Selection 결과: {n_before} -> {len(unit_feat_cols)}개")

if sel_report.get("vote_df") is not None:
    vote_df = sel_report["vote_df"]
    print("\n--- 투표 현황 ---")
    for v in sorted(vote_df["votes"].unique(), reverse=True):
        cnt = (vote_df["votes"] == v).sum()
        print(f"  {v}표: {cnt}개")
    print("\n상위 10 features (투표순):")
    print(vote_df.head(10).to_string(index=False))

Zero-variance unit feature 제거: 149개
Feature Selection 입력: 8003개
Feature Selection 파이프라인 시작
입력 feature 수: 8003
방법: ['boruta', 'lgbm_importance', 'null_importance', 'permutation'], 최소 투표: 3
Device: gpu
[Boruta] max_iter=100, max_depth=5, perc=80, device=gpu, n_samples=26,247


## 6. Stage 2: Unit-level 회귀 (True Two-Stage vs Single 비교)

**방법 A - True Two-Stage**:
- Y>0 샘플만으로 회귀 학습 → E[Y|Y>0] 예측
- 최종 예측 = P(Y>0) x E[Y|Y>0]

**방법 B - Single Regression** (기존 방식 개선):
- 전체 데이터로 회귀 학습, clf_proba_mean은 feature

**공통**: K-Fold CV (val을 early stopping에 사용하지 않음)

In [ ]:
def run_twostage_regression(unit_data, feat_cols, reg_params, n_folds=N_FOLDS):
    """True Two-Stage: Y>0만으로 회귀 학습, 최종 = proba * reg_pred"""
    X_train = unit_data["train"][feat_cols].values
    y_train = unit_data["train"][TARGET_COL].values
    X_val = unit_data["val"][feat_cols].values
    X_test = unit_data["test"][feat_cols].values

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof_reg = np.zeros(len(X_train))
    fold_models = []

    print(f"  Train: {len(X_train):,}개 (Y>0: {(y_train > 0).sum():,}개)")

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train)):
        pos_tr = y_train[tr_idx] > 0
        pos_va = y_train[va_idx] > 0

        reg = lgb.LGBMRegressor(**reg_params)
        reg.fit(
            X_train[tr_idx][pos_tr], y_train[tr_idx][pos_tr],
            eval_set=[(X_train[va_idx][pos_va], y_train[va_idx][pos_va])],
            callbacks=[lgb.early_stopping(REG_EARLY_STOP, verbose=False),
                       lgb.log_evaluation(0)],
        )

        oof_reg[va_idx] = reg.predict(X_train[va_idx])
        fold_models.append(reg)
        print(f"  Fold {fold+1}: train_pos={pos_tr.sum():,}, "
              f"val_pos={pos_va.sum():,}, best_iter={reg.best_iteration_}")

    val_reg = np.mean([m.predict(X_val) for m in fold_models], axis=0)
    test_reg = np.mean([m.predict(X_test) for m in fold_models], axis=0)

    proba_train = unit_data["train"]["clf_proba_mean"].values
    proba_val = unit_data["val"]["clf_proba_mean"].values
    proba_test = unit_data["test"]["clf_proba_mean"].values

    return {
        "val_pred": postprocess(proba_val * val_reg),
        "test_pred": postprocess(proba_test * test_reg),
        "oof_pred": postprocess(proba_train * oof_reg),
        "models": fold_models,
    }


def run_single_regression(unit_data, feat_cols, reg_params, n_folds=N_FOLDS):
    """Single regression: 전체 데이터로 학습, proba는 feature로 포함"""
    X_train = unit_data["train"][feat_cols].values
    y_train = unit_data["train"][TARGET_COL].values
    X_val = unit_data["val"][feat_cols].values
    X_test = unit_data["test"][feat_cols].values

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X_train))
    fold_models = []

    print(f"  Train: {len(X_train):,}개")

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train)):
        reg = lgb.LGBMRegressor(**reg_params)
        reg.fit(
            X_train[tr_idx], y_train[tr_idx],
            eval_set=[(X_train[va_idx], y_train[va_idx])],
            callbacks=[lgb.early_stopping(REG_EARLY_STOP, verbose=False),
                       lgb.log_evaluation(0)],
        )

        oof_pred[va_idx] = reg.predict(X_train[va_idx])
        fold_models.append(reg)
        print(f"  Fold {fold+1}: best_iter={reg.best_iteration_}")

    val_pred = postprocess(np.mean([m.predict(X_val) for m in fold_models], axis=0))
    test_pred = postprocess(np.mean([m.predict(X_test) for m in fold_models], axis=0))

    return {
        "val_pred": val_pred,
        "test_pred": test_pred,
        "oof_pred": postprocess(oof_pred),
        "models": fold_models,
    }

In [ ]:
# ============================================================
# 두 방식 모두 실행 + 비교
# ============================================================
y_val = unit_data["val"][TARGET_COL].values
y_test = unit_data["test"][TARGET_COL].values

print("방법 A: True Two-Stage (Y>0만 회귀)")
print("=" * 50)
result_twostage = run_twostage_regression(unit_data, unit_feat_cols, reg_params)

print(f"\n방법 B: Single Regression (전체 데이터)")
print("=" * 50)
result_single = run_single_regression(unit_data, unit_feat_cols, reg_params)

# --- 비교 ---
print(f"\n{'=' * 60}")
print("최종 RMSE 비교")
print(f"{'=' * 60}")
print("\n[True Two-Stage]")
ts_val_rmse = evaluate(y_val, result_twostage["val_pred"], label="val")
ts_test_rmse = evaluate(y_test, result_twostage["test_pred"], label="test")

print("\n[Single Regression]")
sg_val_rmse = evaluate(y_val, result_single["val_pred"], label="val")
sg_test_rmse = evaluate(y_test, result_single["test_pred"], label="test")

# 더 나은 모델 선택
if ts_val_rmse <= sg_val_rmse:
    best_method = "True Two-Stage"
    best_val_rmse = ts_val_rmse
    best_test_rmse = ts_test_rmse
    best_result = result_twostage
else:
    best_method = "Single Regression"
    best_val_rmse = sg_val_rmse
    best_test_rmse = sg_test_rmse
    best_result = result_single

print(f"\n>>> 최종 선택: {best_method} (val RMSE: {best_val_rmse:.6f})")

## 7. 시각화

In [ ]:
# ============================================================
# 시각화: 두 방법 비교
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (name, result) in enumerate([
    ("True Two-Stage", result_twostage),
    ("Single Regression", result_single),
]):
    pred_val = result["val_pred"]

    axes[row, 0].hist(y_val, bins=50, alpha=0.7, label="actual", color="steelblue")
    axes[row, 0].hist(pred_val, bins=50, alpha=0.7, label="predicted", color="coral")
    axes[row, 0].set_title(f"{name}: Actual vs Predicted")
    axes[row, 0].legend()

    axes[row, 1].hist(pred_val, bins=50, color="coral", edgecolor="black")
    axes[row, 1].set_title(f"{name}: Predicted Distribution")

    axes[row, 2].scatter(y_val, pred_val, alpha=0.3, s=5, color="steelblue")
    max_v = max(y_val.max(), pred_val.max())
    axes[row, 2].plot([0, max_v], [0, max_v], "r--", linewidth=1)
    axes[row, 2].set_xlabel("Actual")
    axes[row, 2].set_ylabel("Predicted")
    axes[row, 2].set_title(f"{name}: Scatter")

plt.tight_layout()
plt.show()

## 8. 실험 로깅

In [ ]:
# ============================================================
# 실험 로깅
# ============================================================
log_experiment(
    exp_id=EXP_ID,
    exp_type=EXP_TYPE,
    best_model=f"LightGBM ({best_method})",
    val_rmse=best_val_rmse,
    test_rmse=best_test_rmse,
    n_features=len(unit_feat_cols),
    memo=f"{EXP_MEMO} | 선택: {best_method} | TwoStage val={ts_val_rmse:.6f} | Single val={sg_val_rmse:.6f}",
    cleaning_params=cleaning_params,
    outlier_params=outlier_params,
    feature_sel_params=selection_params,
    agg_params={
        "agg_funcs": AGG_FUNCS,
        "proba_agg": PROBA_AGG,
    },
    model_params={
        "clf": clf_params,
        "reg": reg_params,
        "n_folds": N_FOLDS,
        "clf_early_stop": CLF_EARLY_STOP,
        "reg_early_stop": REG_EARLY_STOP,
        "clf_threshold": CLF_THRESHOLD,
        "use_sampling": USE_SAMPLING,
        "sample_frac": SAMPLE_FRAC,
    },
    feature_cols=unit_feat_cols,
    xlsx_gdrive_id=XLSX_GDRIVE_ID,
    json_gdrive_id=JSON_GDRIVE_ID,
)